# Duka Electronics · H1 2026 Board Report
## Storyboard (written before any chart is drawn)

**Audience.** The Duka Electronics board: non-technical, short on time, reading on their phones before the meeting.

**Decision they own.** Where to allocate the second-half marketing budget.

**Big Idea.** *Revenue is flat at about KES 36M a quarter, but Mombasa and Nakuru each fell by about a quarter in Q2 while Eldoret grew by half, so the H2 marketing budget should first go to recovering Mombasa and Nakuru.*

**SCR (Situation, Complication, Resolution).**
1. **Situation:** Duka earned KES 71.8M in H1 2026, 36% of it in Nairobi, and total revenue held steady between Q1 (KES 36.0M) and Q2 (KES 35.8M).
2. **Complication:** Beneath the flat total, Mombasa (-25%) and Nakuru (-27%) shrank while Eldoret (+50%) grew, and the losses sit almost entirely in laptop sales (KES 3.8M of the KES 3.9M lost).
3. **Resolution:** Direct the first H2 marketing spend at winning back laptop buyers in Mombasa and Nakuru, learn what worked in Eldoret, and review branch results monthly.

**Charts needed to prove the story, with their insight titles** (titles are calculated in code below):
1. *Nairobi earns 36% of H1 revenue, almost double Mombasa* (sorted bar: the size of the business).
2. *Total revenue was flat: KES 36.0M in Q1 and KES 35.8M in Q2* (line over months: the "situation").
3. *Eldoret grew 50% in Q2; Mombasa and Nakuru fell 25% and 27%* (sorted bar of % change: the "complication").
4. *Laptops explain the swings: Mombasa and Nakuru lost KES 3.8M of laptop revenue* (change in revenue, laptops vs other products: the "why").

**Colour rule.** Grey is the default; **red (#8E0000) means a branch to recover** (Mombasa, Nakuru) and **green (#0BA184) the fastest-growing branch** (Eldoret), consistently across charts.

In [1]:
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams["figure.dpi"] = 60
RED, GREEN, GREY, LIGHT, INK, MID = "#8E0000", "#0BA184", "#B5B5B5", "#D9D9D9", "#222222", "#666666"
SOURCE = "Source: Duka POS system, 1 Jan-29 Jun 2026 (1,000 orders)"

sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["quarter"] = np.where(sales["order_date"].dt.month <= 3, "Q1", "Q2")   # Q1 = Jan-Mar, Q2 = Apr-Jun
print("last order date:", sales["order_date"].max().date(), "| total revenue:", f"KES {sales['revenue'].sum():,.0f}")

last order date: 2026-06-29 | total revenue: KES 71,792,500


## Part A · Chart makeover: before and after
The same data (H1 revenue by product), first as Matplotlib draws it by default, then redesigned. Both versions are saved to `figures/`.

In [2]:
revenue_by_product = sales.groupby("product")["revenue"].sum()

# BEFORE: default, no styling
fig, ax = plt.subplots()
revenue_by_product.plot(kind="bar", ax=ax)
fig.savefig("figures/before.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# AFTER: sorted, horizontal, decluttered, one highlight, direct labels, calculated insight title
ranked = revenue_by_product.sort_values()                         # ascending so the largest bar is on top
top, share = ranked.index[-1], ranked.iloc[-1] / ranked.sum() * 100
others_share = 100 - share
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(ranked.index, ranked / 1e6, color=[RED if p == top else GREY for p in ranked.index], height=0.65)
for sp in ["top", "right", "bottom"]:
    ax.spines[sp].set_visible(False)
ax.xaxis.set_visible(False)
ax.tick_params(axis="y", length=0, labelsize=11)
for y, v in enumerate(ranked / 1e6):
    ax.text(v + 0.6, y, f"KES {v:.1f}M", va="center", fontsize=11, color=INK, fontweight="bold" if ranked.index[y] == top else "normal")
ax.set_xlim(0, ranked.max() / 1e6 * 1.22)
title = f"{top}s earn {share:.0f}% of revenue, more than all other products combined" if share > others_share else f"{top}s earn {share:.0f}% of revenue"
ax.set_title(textwrap.fill(title, 55), loc="left", fontsize=13, fontweight="bold", color=INK, pad=14)
ax.text(0, -0.1, "H1 2026 revenue by product, KES millions  |  " + SOURCE, transform=ax.transAxes, fontsize=8, color=MID)
fig.savefig("figures/after.png", dpi=150, bbox_inches="tight")
plt.close(fig)

### Side by side

| Before: Matplotlib default | After: redesigned |
|---|---|
| ![before](figures/before.png) | ![after](figures/after.png) |

**What changed and why**

| Design change | Before | After | Reason |
|---|---|---|---|
| **Sorted bars** | Alphabetical order | Largest to smallest | The eye can rank products instantly; alphabetical order carries no meaning. |
| **Horizontal bars** | Vertical, tilted labels | Horizontal, readable names | Product names are long; horizontal bars need no rotated text and read well on a phone. |
| **Spines and axis removed** | Box, tick marks and a y-axis in scientific units | Top, right and bottom spines and the value axis removed | Every non-data element was removed; the numbers are printed on the bars, so the axis is redundant. |
| **Colour with purpose** | Default blue for every bar | All grey except the top product in redberry (#8E0000) | Colour now means one thing: *this is the product to look at*. |
| **Direct value labels** | Reader must estimate from the axis | "KES 41.5M" printed at the end of each bar | Removes lookup effort and avoids misreading. |
| **Insight title (calculated in code)** | None | "Laptops earn 58% of revenue, more than all other products combined" | The title states the conclusion; because it is built from the data, it can never go stale or disagree with the chart. |
| **Units and source** | None | Units and a source line under the chart | The chart still makes sense if it is forwarded without any explanation. |

## Part B · The one-page board report
Four charts, built from small summary tables (aggregate first, then plot), laid out in a single tall column that reads well on a phone.

In [3]:
# ---- Summary tables ----
total_rev = sales["revenue"].sum()
branch_rev = sales.groupby("branch")["revenue"].sum().sort_values(ascending=False)

monthly = sales.groupby(sales["order_date"].dt.month)["revenue"].sum()
quarter_rev = sales.groupby("quarter")["revenue"].sum()

by_q = sales.pivot_table(index="branch", columns="quarter", values="revenue", aggfunc="sum")
pct_change = ((by_q["Q2"] / by_q["Q1"] - 1) * 100).sort_values(ascending=False)

laptop_q = sales[sales["product"] == "Laptop"].pivot_table(index="branch", columns="quarter", values="revenue", aggfunc="sum")
d_laptop = laptop_q["Q2"] - laptop_q["Q1"]
lap_n = sales[sales["product"] == "Laptop"].pivot_table(index="branch", columns="quarter", values="order_id", aggfunc="count")   # laptop orders
d_all = by_q["Q2"] - by_q["Q1"]
d_other = d_all - d_laptop

focus = ["Mombasa", "Nakuru"]
BRANCH_COLOR = {b: (RED if b in focus else GREEN if b == "Eldoret" else GREY) for b in by_q.index}
loss_all = -d_all[focus].sum(); loss_laptop = -d_laptop[focus].sum()

print((by_q / 1e6).round(2).assign(change_pct=pct_change.round(1)))
print("\nChange in revenue Q1->Q2 (KES M): laptops vs other products")
print((pd.DataFrame({"laptops": d_laptop, "other": d_other, "total": d_all}) / 1e6).round(2))

quarter     Q1     Q2  change_pct
branch                           
Eldoret   3.54   5.31        49.9
Kisumu    5.01   5.86        17.0
Mombasa   8.02   5.98       -25.5
Nairobi  12.48  13.62         9.1
Nakuru    6.90   5.07       -26.5

Change in revenue Q1->Q2 (KES M): laptops vs other products
         laptops  other  total
branch                        
Eldoret     2.28  -0.51   1.77
Kisumu      1.04  -0.19   0.85
Mombasa    -2.08   0.04  -2.04
Nairobi     1.10   0.03   1.14
Nakuru     -1.76  -0.08  -1.83


In [4]:
def style(ax, title, xlabel):
    """Shared chart styling: insight title, units label, source line, minimal spines."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#999999")
    ax.set_title(textwrap.fill(title, 44), loc="left", fontsize=12.5, fontweight="bold", color=INK, pad=10)
    ax.set_xlabel(xlabel, fontsize=9.5, color=MID, loc="left")
    ax.tick_params(axis="both", labelsize=10, colors=INK, length=3)
    ax.annotate(SOURCE, xy=(0, 0), xycoords="axes fraction", xytext=(0, -44), textcoords="offset points", fontsize=8, color=MID, ha="left", va="top")

**Chart 1: size of the business** (sorted bar; Mombasa and Nakuru in red, Eldoret in green)

In [5]:
def chart_1(ax):
    r = branch_rev.sort_values()
    ax.barh(r.index, r / 1e6, color=[BRANCH_COLOR[b] for b in r.index], height=0.65)
    for y, (b, v) in enumerate(r.items()):
        ax.text(v / 1e6 + 0.4, y, f"KES {v / 1e6:.1f}M ({v / total_rev * 100:.0f}%)", va="center", fontsize=9.5, color=INK)
    ax.set_xlim(0, r.max() / 1e6 * 1.5)
    style(ax, f"{branch_rev.index[0]} earns {branch_rev.iloc[0] / total_rev * 100:.0f}% of H1 revenue, almost double {branch_rev.index[1]}", "H1 2026 revenue (KES millions)")
    ax.spines[["left"]].set_visible(False); ax.tick_params(axis="y", length=0)

**Chart 2: total revenue over time** (line; June is hollow because the data end on 29 June)

In [6]:
def chart_2(ax):
    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun*"]
    ax.plot(months, monthly.values / 1e6, color=INK, linewidth=2.2, marker="o", markersize=6)
    ax.plot(["Jun*"], [monthly.values[-1] / 1e6], marker="o", markersize=7, markerfacecolor="white", color=INK)
    for x, v in zip(months, monthly.values / 1e6):
        ax.text(x, v + 0.9, f"{v:.1f}", ha="center", fontsize=9.5, color=INK)
    ax.axvline(2.5, color="#999999", linestyle=":", linewidth=1.2)
    ax.text(1, 4.5, f"Q1 total\nKES {quarter_rev['Q1'] / 1e6:.1f}M", ha="center", va="center", fontsize=10.5, color=INK, fontweight="bold")
    ax.text(4, 4.5, f"Q2 total\nKES {quarter_rev['Q2'] / 1e6:.1f}M", ha="center", va="center", fontsize=10.5, color=INK, fontweight="bold")
    ax.set_ylim(0, 18)
    style(ax, f"Total revenue was flat: KES {quarter_rev['Q1'] / 1e6:.1f}M in Q1 and KES {quarter_rev['Q2'] / 1e6:.1f}M in Q2", "Monthly revenue (KES millions); *June data end 29 June")

**Chart 3: where the change happened** (sorted diverging bar of Q2 vs Q1 % change)

In [7]:
def chart_3(ax):
    p = pct_change.sort_values()
    ax.barh(p.index, p.values, color=[BRANCH_COLOR[b] for b in p.index], height=0.65)
    for y, v in enumerate(p.values):
        ax.text(v + (2.5 if v >= 0 else -2.5), y, f"{v:+.0f}%", va="center", ha="left" if v >= 0 else "right", fontsize=10, color=INK, fontweight="bold")
    ax.axvline(0, color="#777777", linewidth=1)
    ax.set_xlim(p.min() * 1.5, p.max() * 1.25)
    style(ax, f"Eldoret grew {pct_change['Eldoret']:.0f}% in Q2; Mombasa and Nakuru fell {abs(pct_change['Mombasa']):.0f}% and {abs(pct_change['Nakuru']):.0f}%", "Change in revenue, Q2 vs Q1 (%)")
    ax.spines[["left"]].set_visible(False); ax.tick_params(axis="y", length=0)

**Chart 4: why it happened** (change in revenue split into laptops and all other products)

In [8]:
def chart_4(ax):
    order = pct_change.sort_values().index.tolist()
    y = np.arange(len(order)); h = 0.36
    ax.barh(y + h / 2, d_laptop[order] / 1e6, height=h, color="#444444", label="Laptops")
    ax.barh(y - h / 2, d_other[order] / 1e6, height=h, color=LIGHT, label="All other products")
    for yy, b in zip(y, order):
        for off, v in ((h / 2, d_laptop[b]), (-h / 2, d_other[b])):
            ax.text(v / 1e6 + (0.08 if v >= 0 else -0.08), yy + off, (f"{v / 1e6:+.1f}" if abs(v) >= 5e4 else "0.0"), va="center", ha="left" if v >= 0 else "right", fontsize=9, color=INK)
    ax.set_yticks(y); ax.set_yticklabels(order)
    ax.axvline(0, color="#777777", linewidth=1)
    ax.set_xlim(-3.1, 3.1)
    ax.legend(loc="upper left", fontsize=9.5, frameon=False)
    style(ax, f"Laptops explain the swings: Mombasa and Nakuru lost KES {loss_laptop / 1e6:.1f}M of laptop revenue", "Change in revenue, Q2 vs Q1 (KES millions)")
    ax.spines[["left"]].set_visible(False); ax.tick_params(axis="y", length=0)

**Assemble the page and export it** (header with the Big Idea, the four charts, the recommendation box; saved as PNG and PDF)

In [9]:
fig = plt.figure(figsize=(6.5, 18.6))
gs = fig.add_gridspec(6, 1, height_ratios=[1.7, 2.2, 2.2, 2.3, 2.5, 3.0], hspace=0.78, left=0.2, right=0.93, top=0.99, bottom=0.012)

# Header: the Big Idea
ax = fig.add_subplot(gs[0]); ax.axis("off")
big_idea = ("Revenue is flat, but Mombasa and Nakuru fell about a quarter while Eldoret grew by half. "
            "Put H2 marketing behind recovering Mombasa and Nakuru.")
ax.text(-0.17, 1.0, "DUKA ELECTRONICS  |  H1 2026 BOARD REPORT", fontsize=9, color=MID, fontweight="bold", va="top", transform=ax.transAxes)
ax.text(-0.17, 0.80, textwrap.fill(big_idea, 46), fontsize=13.5, color=INK, fontweight="bold", va="top", transform=ax.transAxes, linespacing=1.25)
ax.text(-0.17, -0.12, "Colour key:  red = branch to recover   green = fastest-growing branch   grey = other branches", fontsize=8.5, color=MID, transform=ax.transAxes)

# The four charts
for chart, row in ((chart_1, 1), (chart_2, 2), (chart_3, 3), (chart_4, 4)):
    chart(fig.add_subplot(gs[row]))

# Recommendation box
ax = fig.add_subplot(gs[5]); ax.axis("off")
ax.add_patch(FancyBboxPatch((-0.17, -0.02), 1.19, 1.0, boxstyle="round,pad=0.0,rounding_size=0.02", transform=ax.transAxes, facecolor="#F4F1F1", edgecolor="none", clip_on=False))
rec = ("WHAT WE RECOMMEND FOR H2\n"
       f"1. Put the first marketing money into winning back laptop buyers in Mombasa and Nakuru (KES {loss_all / 1e6:.1f}M lost in Q2).\n"
       f"2. Find out what lifted Eldoret's laptop sales ({int(lap_n.loc['Eldoret', 'Q1'])} to {int(lap_n.loc['Eldoret', 'Q2'])} orders) and test it elsewhere.\n"
       "3. Review branch sales every month.\n"
       "Caution: each branch has 50-200 orders a quarter, so part of these moves may be chance; Q2 data end on 29 June.")
wrapped = "\n".join(textwrap.fill(l, 60, subsequent_indent="    ") if not l.startswith("WHAT") else l for l in rec.split("\n"))
ax.text(-0.14, 0.93, wrapped, fontsize=9.5, color=INK, va="top", transform=ax.transAxes, linespacing=1.4)

fig.savefig("reports/duka_board_report.png", dpi=200)
fig.savefig("reports/duka_board_report.pdf")
plt.close(fig)
print("saved: reports/duka_board_report.png and reports/duka_board_report.pdf")

saved: reports/duka_board_report.png and reports/duka_board_report.pdf


### The finished one-page report
![Duka Electronics H1 2026 board report](reports/duka_board_report.png)

In [10]:
# Does each chart actually support its title? Re-derive each claim from the data and assert it.
claims = {
    "1. Nairobi earns about 36% of H1 revenue, almost double Mombasa": (round(branch_rev["Nairobi"] / total_rev * 100), round(branch_rev["Nairobi"] / branch_rev["Mombasa"], 2)),
    "2. Q1 and Q2 totals are flat (difference, %)": round((quarter_rev["Q2"] / quarter_rev["Q1"] - 1) * 100, 2),
    "3. Eldoret +50%, Mombasa -25%, Nakuru -27%": (round(pct_change["Eldoret"]), round(pct_change["Mombasa"]), round(pct_change["Nakuru"])),
    "4. Laptops' share of the Mombasa + Nakuru loss (%)": round(loss_laptop / loss_all * 100),
}
for k, v in claims.items():
    print(k, "->", v)
assert round(branch_rev["Nairobi"] / total_rev * 100) == 36 and branch_rev["Nairobi"] / branch_rev["Mombasa"] > 1.8
assert abs(quarter_rev["Q2"] / quarter_rev["Q1"] - 1) < 0.01
assert pct_change["Eldoret"] > 45 and pct_change["Mombasa"] < -20 and pct_change["Nakuru"] < -20
assert loss_laptop / loss_all > 0.9
print("\nAll four titles are supported by the data.")

1. Nairobi earns about 36% of H1 revenue, almost double Mombasa -> (36, np.float64(1.86))
2. Q1 and Q2 totals are flat (difference, %) -> -0.33
3. Eldoret +50%, Mombasa -25%, Nakuru -27% -> (50, -25, -27)
4. Laptops' share of the Mombasa + Nakuru loss (%) -> 99

All four titles are supported by the data.


**Check result.** All four titles hold when re-derived from the data: Nairobi's share and its lead over Mombasa (chart 1), the near-identical quarter totals (chart 2), the three growth/decline figures (chart 3) and laptops' share of the Mombasa and Nakuru decline (chart 4).

**Design choices in the report.** One column so it scrolls naturally on a phone; every chart has a sentence-style title, a units label and a source line; grey is the default and colour is used only for the branches the decision is about (red and green keep the same meaning in charts 1, 3 and 4); bars are sorted; values are printed directly on the bars; and the recommendation sits at the bottom, where the story ends. The report is saved as `reports/duka_board_report.png` and `.pdf`. **Honest limits are on the page:** the data end on 29 June, and branch samples are small.

## Board Summary (150 words or fewer)

In [11]:
board_summary = (
    'Duka Electronics sold KES 71.8M in the first half of 2026, and Nairobi alone delivered 36% of it '
    'Total sales were steady: KES 36.0M from January to March and KES 35.8M from April to June '
    'Underneath, the picture changed '
    'Eldoret grew by 50%, but Mombasa and Nakuru fell by about a quarter each, losing KES 3.9M between them '
    'Almost all of that loss, KES 3.8M, came from fewer laptop sales, our biggest earner '
    "Their combined loss is more than twice Eldoret's gain of KES 1.8M, and they are our second- and third-largest branches "
    'We recommend that the second-half marketing budget first goes to winning back laptop buyers in Mombasa and Nakuru, and that we learn what Eldoret did right '
    'One caution: each branch has only a few hundred orders, so part of these swings may be chance '
    'We should review branch results every month. '
)
print(len(board_summary.split()), 'words')

144 words


Duka Electronics sold KES 71.8M in the first half of 2026, and Nairobi alone delivered 36% of it. Total sales were steady: KES 36.0M from January to March and KES 35.8M from April to June. Underneath, the picture changed. Eldoret grew by 50%, but Mombasa and Nakuru fell by about a quarter each, losing KES 3.9M between them. Almost all of that loss, KES 3.8M, came from fewer laptop sales, our biggest earner. Their combined loss is more than twice Eldoret's gain of KES 1.8M, and they are our second- and third-largest branches. We recommend that the second-half marketing budget first goes to winning back laptop buyers in Mombasa and Nakuru, and that we learn what Eldoret did right. One caution: each branch has only a few hundred orders, so part of these swings may be chance. We should review branch results every month.

## Dashboard data export (for Looker Studio)
ISO dates (YYYY-MM-DD) so that Looker Studio types `order_date` as a Date; `revenue` is numeric.

In [12]:
dash = sales[["order_id", "order_date", "branch", "product", "channel", "quantity", "unit_price", "revenue"]].copy()
dash["order_date"] = dash["order_date"].dt.strftime("%Y-%m-%d")
dash.to_csv("duka_dashboard_data.csv", index=False)
print(dash.shape); print(dash.dtypes); dash.head()

(1000, 8)
order_id      int64
order_date      str
branch          str
product         str
channel         str
quantity      int64
unit_price    int64
revenue       int64
dtype: object


,order_id,order_date,branch,product,channel,quantity,unit_price,revenue
0,1001,2026-06-03,Nakuru,Phone,In-store,3,15000,45000
1,1002,2026-02-02,Nairobi,Laptop,In-store,4,65000,260000
2,1003,2026-01-05,Nakuru,Tablet,In-store,5,28000,140000
3,1004,2026-04-26,Nairobi,Headphones,In-store,5,2500,12500
4,1005,2026-03-07,Nairobi,Headphones,In-store,3,2500,7500
